# Extra: Delta Lake, o Parquet com histórico

Este notebook é um **extra**, fora dos vídeos: as células já vêm preenchidas, para você rodar e ler com calma. O
projeto continua gravando em Parquet; aqui você vê o próximo passo.

## O problema

O Parquet resolveu tamanho e velocidade, mas não tem **transação**. Se um `overwrite` falhar no meio, parte do dado
pode se perder, e não existe "desfazer".

O **Delta Lake** resolve isso. O dado continua sendo **arquivos Parquet**; o que muda é uma pasta extra,
`_delta_log/`, com um registro de cada operação feita na tabela. Com esse registro, o Delta oferece:

- **escrita atômica**: ou a operação termina inteira, ou é como se não tivesse acontecido;
- **time travel**: ler a tabela como ela estava numa versão anterior;
- **`UPDATE`, `DELETE` e `MERGE`**: alterar linhas, o que no Parquet puro exige reescrever tudo.

## Uma sessão com o Delta

O Delta não vem dentro do Spark: é um pacote à parte, e a sessão precisa **nascer** sabendo dele. Por isso este
notebook tem a sua própria sessão. Três novidades em relação à sessão de sempre:

- `spark.sql.extensions` liga os comandos SQL do Delta (como `DESCRIBE HISTORY`);
- `spark.sql.catalog.spark_catalog` troca o catálogo padrão por um que entende tabelas Delta;
- `configure_spark_with_delta_pip` acrescenta ao construtor o pacote do Delta na versão certa para o Spark instalado.

Se aparecer `ClassNotFoundException`, é porque outra sessão já tinha sido aberta neste kernel: use **Kernel > Restart
Kernel** e rode de novo a partir da primeira célula.

In [ ]:
import sys
sys.path.insert(0, "../../src")

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession, functions as F

construtor = (
    SparkSession.builder
    .appName("arena-delta")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(construtor).getOrCreate()

RAW = "../../data/raw"
OUT = "../../data/out"
spark

## Versões de uma tabela

Vamos gravar as partidas em Delta e, em seguida, sobrescrever a tabela só com as Copas a partir de 1990: o mesmo tipo
de `overwrite` que, no Parquet, apagaria o histórico para sempre.

In [ ]:
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas

partidas = limpar_partidas(
    spark.read.option("header", "true").schema(MATCHES).csv(f"{RAW}/WorldCupMatches.csv")
)

caminho_delta = f"{OUT}/partidas_delta"

partidas.write.format("delta").mode("overwrite").save(caminho_delta)                                 # versão 0
partidas.filter(F.col("ano") >= 1990).write.format("delta").mode("overwrite").save(caminho_delta)  # versão 1

!ls {caminho_delta}

In [ ]:
from pathlib import Path

atual = spark.read.format("delta").load(caminho_delta)
versao_0 = spark.read.format("delta").option("versionAsOf", 0).load(caminho_delta)

print("versão atual:", atual.count(), "partidas")
print("versão 0    :", versao_0.count(), "partidas")

spark.sql(f"DESCRIBE HISTORY delta.`{Path(caminho_delta).resolve()}`").select("version", "operation").show()

O `overwrite` da versão 1 **não apaga** a versão 0: o Delta guarda as duas, e o `versionAsOf` volta no tempo. É a
rede de segurança que o Parquet puro não tem.

**Quando usar:** quando a tabela precisa de atualização de linhas, de histórico de versões ou de proteção contra falhas
no meio da escrita. Para o que o projeto precisa hoje (carga por Copa, com o overwrite dinâmico do vídeo 5.3), o
Parquet basta.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()